# 🚀 Qari Finder: Cloud Trainer (Drive → Colab)

This notebook:
- Mounts Google Drive
- Copies your synced project from Drive to Colab VM (fast disk)
- Installs requirements
- Runs prepare → train → eval → convert
- Syncs results back to Drive

✅ Tip: Update `DRIVE_ROOT` below if your Drive folder is different.

In [ ]:
# @title Mount Drive + Sync project to VM
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# ===== CONFIG (EDIT THIS IF NEEDED) =====
# This should point to the SAME folder your PC sync script pushes into.
DRIVE_ROOT = "/content/drive/MyDrive/ML/qari"  # e.g. .../MyDrive/ML/qari

# Fast local disk on the Colab VM
LOCAL_ROOT = "/content/work"
RESEARCH_DIR = f"{LOCAL_ROOT}/research"
APP_MODELS_DIR = f"{LOCAL_ROOT}/app/public/models"

# Optional: start fresh each run
CLEAN_LOCAL = False

if not os.path.exists(DRIVE_ROOT):
    raise FileNotFoundError(
        f"DRIVE_ROOT not found: {DRIVE_ROOT}\n"
        "Fix DRIVE_ROOT to match your Drive folder."
    )

if CLEAN_LOCAL and os.path.exists(LOCAL_ROOT):
    !rm -rf "{LOCAL_ROOT}"

!mkdir -p "{LOCAL_ROOT}"

print("⏳ Syncing Drive -> VM (fast disk) ...")
!rsync -a \
  --exclude 'venv' \
  --exclude '__pycache__' \
  --exclude '.git' \
  --exclude '.idea' \
  --exclude 'node_modules' \
  --exclude '.DS_Store' \
  "{DRIVE_ROOT}/" "{LOCAL_ROOT}/"

# Ensure expected folders exist
os.makedirs(APP_MODELS_DIR, exist_ok=True)

print("✅ Local project ready")
print("✅ LOCAL_ROOT    =", LOCAL_ROOT)
print("✅ RESEARCH_DIR  =", RESEARCH_DIR)
print("✅ APP_MODELS_DIR=", APP_MODELS_DIR)

# Quick sanity check
if not os.path.exists(RESEARCH_DIR):
    raise FileNotFoundError(
        f"Expected research folder at {RESEARCH_DIR} but it's missing.\n"
        "Make sure your Drive has: ML/qari/research (from your push script)."
    )


In [ ]:
# @title 1️⃣ Create Isolated Environment
import os

# 1. Create the venv (allowing it to be "pip-less" for now)
print("🛠️ Creating bare virtual environment...")
!python3 -m venv /content/project_env --without-pip

VENV_PYTHON = "/content/project_env/bin/python3"
!{VENV_PYTHON} --version

# 2. Download the standalone pip installer
print("📥 Downloading standalone pip installer...")
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py

# 2. Install system-level audio decoders (Colab's system needs these for MP3)
!apt-get install -y ffmpeg -qq

# 3. Use the venv's internal python to install pip into itself
print("💉 Injecting pip into the environment...")
!/content/project_env/bin/python3 get-pip.py
os.remove("get-pip.py")

# 4. Verify the link
VENV_PIP = "/content/project_env/bin/pip"
if os.path.exists(VENV_PIP):
    print("\n✅ SUCCESS: Pip is now active in /content/project_env/bin/pip")
    !{VENV_PIP} --version
else:
    print("\n❌ FAILED: Pip was not injected correctly.")

# 5. NOW install your requirements safely
print(f"\n📦 Installing requirements from {RESEARCH_DIR}...")
req = f"{RESEARCH_DIR}/requirements.txt"
if not os.path.exists(req):
    raise FileNotFoundError(f"requirements.txt not found at: {req}")
!{VENV_PIP} install -r "{req}"

!{VENV_PYTHON} -c "import tensorflow as tf; print('='*30); print('✅ GPU:', tf.config.list_physical_devices('GPU')); print('📦 TF:', tf.__version__); print('='*30)"

if os.path.exists(RESEARCH_DIR):
    %cd {RESEARCH_DIR}
    print(f"✅ Now in: {os.getcwd()}")
else:
    print(f"❌ Error: {RESEARCH_DIR} not found!")

In [ ]:
# @title 3) Prepare data
!{VENV_PYTHON} prepare_data.py


In [ ]:
# @title 4) Train
!{VENV_PYTHON} train.py


In [ ]:
# @title 🧪 Run Export & Parity Test
import os

# 1. Update the Audio Physics (JSON)
print("📤 Exporting Audio Config...")
!{VENV_PYTHON} tools/export_ears.py

# 2. Run the Parity Math Check
print("\n🔍 Running Matrix Parity Test...")
!{VENV_PYTHON} tools/verify_matrix.py

print("\n✅ Done. Compare the 'Python Matrix Results' above with your Local PC and App UI.")

In [ ]:
# @title 5) Evaluate
!{VENV_PYTHON} evaluate_model.py


In [ ]:
# @title 6) Convert
# 1. Install the converter tool first!
!pip install tensorflowjs

# 2. Run the conversion
# (Using the 'tf_saved_model' format which is the most stable)
print("🔄 Starting Conversion...")
!tensorflowjs_converter --input_format=tf_saved_model --output_node_names='output_node' ./models/qari_model_export ../app/public/models/tfjs_model

print("🎉 Conversion Complete! Check the 'tfjs_model' folder.")

In [ ]:
# @title 7) Sync results back to Drive
import os
import shutil
from pathlib import Path

# Ensure destination folders exist on Drive
os.makedirs(f"{DRIVE_ROOT}/research/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/app/public/models", exist_ok=True)

# A) Copy the newest .h5 from local models/ (more robust than hardcoding a filename)
models_dir = Path("models")
h5_files = sorted(models_dir.glob("*.h5"), key=lambda p: p.stat().st_mtime, reverse=True)

if not h5_files:
    raise FileNotFoundError("No .h5 files found in local 'models/' folder.")

latest_h5 = h5_files[0]
dst_h5 = Path(DRIVE_ROOT) / "research" / "models" / latest_h5.name
shutil.copy2(latest_h5, dst_h5)
print("✅ Saved H5 to Drive:", dst_h5)

# B) Sync TFJS web model files back to Drive
print("⏳ Syncing TFJS files back to Drive...")
!rsync -a "{APP_MODELS_DIR}/" "{DRIVE_ROOT}/app/public/models/"
print("✅ TFJS synced to Drive")


In [ ]:
# @title 🔄 REFRESH: Pull Latest Code from Drive
import os

print("⏳ Syncing Drive -> VM (Updating changed files)...")

# 1. Force Sync (Update changed files, delete files removed from Drive)
# --delete ensures if you delete a bad file on PC, it gets deleted here too.
!rsync -a --delete \
  --exclude 'venv' \
  --exclude '__pycache__' \
  --exclude '.git' \
  --exclude '.idea' \
  --exclude 'node_modules' \
  --exclude 'datasets/' \
  --exclude '.DS_Store' \
  --exclude 'models/' \
  "{DRIVE_ROOT}/" "{LOCAL_ROOT}/"

print("✅ Code updated! You can now run the next cells.")

# Optional: Print the requirements file to PROVE it updated
print("\n👀 verifying requirements.txt content:")
!cat {RESEARCH_DIR}/requirements.txt | head -n 5